In [13]:
import pandas as pd
import torch
import torch.nn as nn # just get that all up in here. 
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from torch import optim

from torch.optim.lr_scheduler import StepLR
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader

# for toy model, use a SUBSET so we aren't tearing our hair out trying to run this. 
from torch.utils.data import Subset
# same thing that we did for the MLP
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             classification_report, fbeta_score)
import numpy as np



In [14]:
# Constants
projections_path = 'IU-xray/indiana_projections.csv'
# this contains the labelling we want for our image CNN
reports_path = 'IU-xray/indiana_reports.csv'
# and of course, the images
image_dir = 'IU-xray/images/images_normalized'

In [ ]:
# # DF creation. 
# def build_master_df(projections_path, reports_path):
#     proj = pd.read_csv(projections_path)
#     rep  = pd.read_csv(reports_path)

#     proj = proj[proj["projection"] == "Frontal"] # only grab the frontal ones. don't need the other ones. 

#     # CNN only needs uid, filename, and Problems for labeling
#     df = proj.merge(rep[["uid", "Problems"]], on="uid", how="inner")

#     def label_from_text(row):
#         text = str(row.get("Problems", "")).lower().strip()
#         return 0 if text == "normal" else 1

#     df["abnormal_label"] = df.apply(label_from_text, axis=1)

#     # return df[["uid", "filename", "abnormal_label"]]  # nothing else needed
#     # we don't need that uid. 
#     return df[["filename", "abnormal_label"]]  # nothing else needed


# # DF creation. 
# # this is still technically wrong, I want to change from abnormal to pneumonia. There's a lot of other stuff that needs to happen too for that to work.
# def build_master_df_for_pneumonia(projections_path, reports_path):
#     proj = pd.read_csv(projections_path)
#     rep = pd.read_csv(reports_path)
    
#     proj = proj[proj["projection"] == "Frontal"]
#     df = proj.merge(rep[["uid", "Problems"]], on="uid", how="inner")
    
#     def label_pneumonia(row):
#         text = str(row.get("Problems", "")).lower().strip()
#         # Look for explicit pneumonia mentions
#         pneumonia_keywords = ['pneumonia', 'consolidation', 'infiltrate', 'opacity']
        
#         # This is simplified—real CheXNet used NLP extraction from reports
#         # But for your purpose: 1 if pneumonia-related, 0 otherwise
#         if any(keyword in text for keyword in pneumonia_keywords):
#             return 1
#         return 0
    
#     df["pneumonia_label"] = df.apply(label_pneumonia, axis=1)
    
#     # You can drop uid entirely if you want—it's not needed
#     return df[["filename", "pneumonia_label"]]


import re
# USE BETTER VERSION OF THE PNEUMONIA LABEL CREATOR
def simple_negation_check(text, target_terms):
    """Return True if any target term is present and not negated."""
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    
    if not any(term in text_lower for term in target_terms):
        return False
    
    # Negation patterns (window of up to 5 words)
    neg_patterns = [
        r'\bno\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bwithout\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bfree\s+of\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnegative\s+for\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnot\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
    ]
    for pat in neg_patterns:
        if re.search(pat, text_lower):
            return False
    return True

PNEUMONIA_TERMS = ['pneumonia', 'consolidation', 'infiltrate', 'airspace disease', 'opacity']

def build_master_df(projections_path, reports_path):
    proj = pd.read_csv(projections_path)
    rep = pd.read_csv(reports_path)
    
    # Create label column directly on rep
    rep["pneumonia_label"] = rep.apply(
        lambda row: 1 if simple_negation_check(
            f"{row.get('findings','')} {row.get('impression','')}",
            PNEUMONIA_TERMS
        ) else 0,
        axis=1
    )
    
    # Now merge only what's needed
    proj = proj[proj["projection"] == "Frontal"]
    df = proj.merge(rep[['uid', 'pneumonia_label']], on='uid', how='inner')
    
    return df[["filename", "pneumonia_label"]]

In [ ]:
# Dataset creation, dataloader creation as well. 

class ChestXRayDataset(Dataset):
  def __init__(self, records, image_dir, transform=None):
    """
    records    : a dataframe slice (post train test split)
    image_dir : path to folder containing all .dcm.png files
    transform : torchvision transforms pipeline
    """
    self.records = records.reset_index(drop=True)
    self.image_dir = image_dir
    self.transform = transform

  def __len__(self):
    return len(self.records)

  def __getitem__(self, idx):
    row = self.records.iloc[idx]

    # for image stuff
    img_path = os.path.join(self.image_dir, row['filename'])
    image = Image.open(img_path).convert('L') # Make sure to keep this in Black and white, rather than anything else. 
    if self.transform:
      image = self.transform(image)

    # label time
    label = torch.tensor(row["pneumonia_label"], dtype=torch.float32)

    return image, label


# lets try just a humungous batch_size
# there is a function down later that is attempting to derive the batch size. 
# halved the batch size so I could double the image size. 
def get_dataloaders(projections_path, reports_path, image_dir, test_size=0.2,
                    batch_size=128, image_size=224, num_workers=0, subset=False):
  # need to use 224 for 2 reasons: its what they used in the paper, and
  # its what ImageNet is trained on, so we need to ensure compatability. 

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  
  # this is wrong now. I need to fix this. 
  df = build_master_df(projections_path, reports_path)

  train_df, test_df = train_test_split(
      df,
      test_size=test_size,
      random_state=42,
      stratify=df["abnormal_label"] # make sure to get a good split. 
  )

  # DOING some standard imageNet transformations
  train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
    # ok that is for COLORED images. I am workign in black and white, and according to my folders, they are already noramlized. 
    # ok what does this mean and why do pycharm and claude agree thats what the values should be
    # transforms.Normalize(mean=[0.485, 0.456, 0.406],
    #                      std=[0.229, 0.224, 0.225])
  ])

  test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
  ])

  train_dataset = ChestXRayDataset(train_df, image_dir, transform=train_transforms)
  test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)


  # go aheand and just do this here...
  n_neg = (train_df['abnormal_label'] == 0).sum()
  n_pos = (train_df['abnormal_label'] == 1).sum()
  pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

  # if we wnat a subset, just create smaller ones, otherwise do nothing. 
  #  just rip this completely out. 
  # if subset == True:
  #   train_dataset = Subset(train_dataset, range(200))
  #   test_dataset = Subset(test_dataset, range(50))


  train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False  # speeds up GPU transfer... supposedly. 
  )

  test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,   # ← never shuffle test
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False
  )


  # get the df for NLP purposes, will probably find a better way to do it later.
  return train_loader, test_loader, pos_weight

In [17]:
# some data exploration. can take a mintue tho. 
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from PIL import Image

# def explore_image_data(df, image_dir):
#     print(f"Total frontal images : {len(df)}")
#     print(f"Abnormal (1)         : {df['abnormal_label'].sum()}")
#     print(f"Normal   (0)         : {(df['abnormal_label']==0).sum()}")
#     print(f"Class balance        : {df['abnormal_label'].mean():.1%} abnormal")
#     print()

#     # check a few images actually exist and get their sizes
#     sizes = []
#     missing = 0
#     for fname in df['filename'].head(50):
#         path = os.path.join(image_dir, fname)
#         if os.path.exists(path):
#             img = Image.open(path)
#             sizes.append(img.size)
#         else:
#             missing += 1
#             print(f"MISSING: {fname}")

#     print(f"Missing files (first 50): {missing}")
#     print(f"Unique sizes found: {set(sizes)}")

#     # visualize a few normal vs abnormal examples
#     fig, axes = plt.subplots(2, 4, figsize=(16, 8))

#     normal_rows   = df[df['abnormal_label'] == 0].head(4)
#     abnormal_rows = df[df['abnormal_label'] == 1].head(4)

#     for i, (_, row) in enumerate(normal_rows.iterrows()):
#         img = Image.open(os.path.join(image_dir, row['filename']))
#         axes[0][i].imshow(img, cmap='gray')
#         axes[0][i].set_title(f"NORMAL\n{row['filename'][:20]}")
#         axes[0][i].axis('off')

#     for i, (_, row) in enumerate(abnormal_rows.iterrows()):
#         img = Image.open(os.path.join(image_dir, row['filename']))
#         axes[1][i].imshow(img, cmap='gray')
#         axes[1][i].set_title(f"ABNORMAL\n{row['filename'][:20]}")
#         axes[1][i].axis('off')

#     plt.suptitle('Normal vs Abnormal Chest X-Rays', fontsize=14)
#     plt.tight_layout()
#     plt.savefig('data_exploration.png', dpi=150)
#     plt.show()

#     # pixel intensity distribution
#     fig, ax = plt.subplots(figsize=(10, 4))
#     for label, color, name in [(0, 'blue', 'Normal'), (1, 'red', 'Abnormal')]:
#         sample = df[df['abnormal_label'] == label].head(20)
#         all_pixels = []
#         for _, row in sample.iterrows():
#             img = np.array(Image.open(
#                 os.path.join(image_dir, row['filename'])
#             ).convert('L'))
#             all_pixels.extend(img.flatten().tolist())
#         ax.hist(all_pixels, bins=50, alpha=0.5, color=color, label=name, density=True)

#     ax.set_title('Pixel Intensity Distribution: Normal vs Abnormal')
#     ax.set_xlabel('Pixel Value')
#     ax.set_ylabel('Density')
#     ax.legend()
#     plt.savefig('pixel_distribution.png', dpi=150)
#     plt.show()

In [18]:
# create DF
projections_path = 'IU-xray/indiana_projections.csv'
# this contains the labelling we want for our image CNN
reports_path = 'IU-xray/indiana_reports.csv'
# and of course, the images
image_dir = 'IU-xray/images/images_normalized'

df = build_master_df(projections_path, reports_path)
# explore_image_data(df, image_dir)

In [19]:
print(torch.cuda.is_available())       # True/False
print(torch.cuda.get_device_name(0))   # e.g. "NVIDIA GeForce RTX 3060"

True
NVIDIA GeForce RTX 4070


In [20]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

df = build_master_df(projections_path, reports_path)
print("this is what the df looks like \n ", df.head())

Using device: cuda:0
this is what the df looks like 
     uid                   filename  abnormal_label
0    1     1_IM-0001-4001.dcm.png               0
1    2     2_IM-0652-1001.dcm.png               1
2    3     3_IM-1384-1001.dcm.png               0
3    4     4_IM-2050-1001.dcm.png               1
4    5  5_IM-2117-1003002.dcm.png               1


In [21]:
# CNN Class for later usage. 

# ORIGINAL CNN that I cooked up. Need to change for the other thing. 
# We need: 
# so to make a 
# 121 layers (GOOD HEAVENS)
# DENSENET model (GOOD HEAVENS)
# Batch normalization
# replace the final fully connected layer with a single output, 
# and then apply sigmoid nonlineraity. 
# they then used a fine tuned model from ImageNet, then Adam, minbatch of size 16, (???), decayed factor of 10
class ChestCNN(nn.Module):
    def __init__(self):
        super(ChestCNN, self).__init__()

        # lets try 4 blocks. 
        # problem: Not all images are the same size. Far from it, even. 
        # so we need to handle that at some point as well. 

        # FOR NOW, ASSUME INPUT FEATURES are 224. 

        image_size = 224
        self.features = nn.Sequential(
            # block 1. lets focus on low level features. 
            # input: (batch, 1, 224, 224) # batch of BW images (so only 1 channel)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32), 
            nn.ReLU(), # need an activation function somewhere now don't we
            nn.MaxPool2d(kernel_size=2, stride=2),
            # output is now (batch, 32, 112, 112)

            # block 2 -- tecture time
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), 
            nn.BatchNorm2d(64), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2, stride=2),
            # output is now (Batch, 64, 56, 56)

            # block 3, lets focus on larger structures
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1), 
            nn.BatchNorm2d(128), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2, stride=2), 
            # output is now batch, 128, 128, 128

            # block 4 -- global super structure now
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1), 
            nn.BatchNorm2d(256), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=2, stride=2),
            # output: (batch, 256, 14, 14)

        )

        # now we need to collapse each of the 256 to a single number
        # much less overtitting than fletting
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        # output: Batch, 256, 1, 1

        self.classifier = nn.Sequential(
            nn.Flatten(), # batch, 256
            nn.Linear(256, 128), 
            nn.ReLU(), 
            nn.Dropout(p=0.5), # thats really high. Might want to tune down. 
            nn.Linear(128, 1), # compress it all down to a single value
            # and no sigmoid! Remember we are using BCEWithLogitsLoss (like MLP) so we don't need it. 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x



# uses Kaiming He initalization for the model. 
def weights_init_cnn(layer):
    if isinstance(layer, nn.Conv2d):
        # we are using kaiming initalization, should be better
        # apparently fan out is better for CNN's 
        nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
        if layer.bias is not None:
            nn.init.constant_(layer.bias, 0)

    elif isinstance(layer, nn.BatchNorm2d):

        nn.init.constant_(layer.weight, 1)
        nn.init.constant_(layer.bias, 0)

    elif isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
        nn.init.constant_(layer.bias, 0)


        

In [22]:
# ---- setup everything for training ----

model = ChestCNN().to(device) # create the actual model. 
model.apply(weights_init_cnn)

# lets start with the CheXNet init params cause why not
learning_rate = 0.0001 # decay by factor of 10 each time validation loss platueas after epoch
# lets also make it 10x smaller, 1e-4 should work better. 

subset = False # this can either make it smaller or bigger depending on our needs. 
# Adam optimizer, standard params. 
train_loader, test_loader, pos_weight = get_dataloaders(projections_path, reports_path, image_dir, subset=subset) # after that I got a bunch of defaults, I'll change em when I get there.

loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # trying a different loss function, we shall see.
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

In [23]:
# fresh model — no loading from checkpoint
model = ChestCNN().to(device)
model.apply(weights_init_cnn)
patience = 20 # stop if there is no improvement after 20 epochs. 

# lower LR from the start
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
# weight_decay adds L2 regularization — helps prevent overfitting
# on a small dataset like yours

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=patience, factor=0.5,
)

# 512x512 from the start
train_loader, test_loader, pos_weight = get_dataloaders(
    projections_path, reports_path, image_dir,
    image_size=512,
    batch_size=128
)

In [24]:
# Training loop time :)

# ---- config ----
num_epochs = 0 # want to not run this rn # just to give us a place to start. 
validate_every = 5
patience = 20 # stop if there is no improvement after 15 epochs. 
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True) # lets save these in case of silent crashing. never hurt to be careful overnight. 

# ---- tracking stuff ---- 
train_losses  = []
test_losses   = []
best_val_loss = float('inf')
best_weights  = None
patience_counter  = 0
epochs_completed  = 0

for epoch in range(num_epochs):
    epochs_completed += 1
    # print("Starting epoch ", epoch)
    # --- training phase ---- 
    model.train()
    epoch_loss = 0.0 # make him float. 

    for i, (images, labels) in enumerate(train_loader):
        
        # we should already have this thrown 
        images = images.to(device, non_blocking=True) # non blocking for even more speed
        labels = labels.to(device, non_blocking=True) # ditto


        outputs = model(images) # take that whole batch in 
        loss = loss_function(outputs.squeeze(1), labels) # ouptuts need to be squeezed to change shape from (B, 1) to just (B)

        # backward and optimize
        optimizer.zero_grad(set_to_none=True) # apparently more efficient than zero_Grad()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # get loss value and clean up
        loss_value = loss.item()
        epoch_loss += loss_value

        del outputs, loss, images, labels # make sure to clean these up to keep the vram nice and neat :)
    
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1:4f} | Train Loss {epoch_loss:.4f}", end="")


    # ---- validation phase ----
    if (epoch + 1) % validate_every == 0:
        model.eval()
        correct =  0 
        total = 0
        total_test_loss = 0
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for test_images, test_labels in test_loader:
                test_images = test_images.to(device, non_blocking=True)
                test_labels = test_labels.to(device, non_blocking=True)

                outputs = model(test_images)
                test_loss = loss_function(outputs.squeeze(1), test_labels)
                probs = torch.sigmoid(outputs.squeeze(1)) # bind between 0 and 1 for the actual probability (sigmoid)
                predicted = (probs >= 0.5).float()  # need to use an actual label for compairson
                total += test_labels.size(0)
                correct += (predicted == test_labels).sum().item()
                total_test_loss += test_loss.item()
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(test_labels.cpu().numpy())

                del outputs, test_loss # lil cleanup action

        
        total_test_loss /= len(test_loader)
        accuracy = correct / total
        test_losses.append(total_test_loss)
        # lets do our AUC
        auc = roc_auc_score(all_labels, all_probs)

        print(f" | Val Loss: {total_test_loss:.4f} | "
              f"Val Acc: {accuracy:.3f} | AUC: {auc:.4f}", end="")

        scheduler.step(total_test_loss) # interesting...


        # ---- check for early stopping ----
        if total_test_loss < best_val_loss - 1e-4: # probably more idiomatic to make this an epsilon. 
            best_val_loss = total_test_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0  # take a deep breath buddy. 
            print(" <- best", end="")

            # now we gotta save the best checkpoint
            torch.save({
                "epoch" : epoch + 1, 
                "model_state_dict" : best_weights, 
                "optimizer_state_dict ": optimizer.state_dict(),
                "val_loss" : best_val_loss, 
                "auc" : auc, 
                "train_losses" : train_losses, 
                "test_losses" : test_losses, 
            }, f"{checkpoint_dir}/best_model.pt")

        else:
            patience_counter += 1
            print(f" | patience {patience_counter}/{patience}", end="")

            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epochs_completed} "
                      f"— best val loss: {best_val_loss:.4f}")
                model.load_state_dict(best_weights)
                break

        if (epoch + 1) % (validate_every * 10) == 0:
            torch.save({
                'epoch':                epoch + 1,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_losses':         train_losses,
                'test_losses':          test_losses,
            }, f"{checkpoint_dir}/checkpoint_epoch_{epoch+1}.pt")
            print(f" | checkpoint saved", end="")

    print()  # newline after each epoch's print



    if device.type == "cuda": # I think?? make sure to clear the memeory.
        torch.cuda.empty_cache()


print(f"\nTraining complete — {epochs_completed} epochs")
print(f"Best val loss: {best_val_loss:.4f}")

# --- load best model for inference ---
checkpoint = torch.load(f"{checkpoint_dir}/best_model.pt")
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} "
      f"with AUC: {checkpoint['auc']:.4f}")


Training complete — 0 epochs
Best val loss: inf
Loaded best model from epoch 100 with AUC: 0.7088


In [25]:
def evaluate_cnn(model, test_loader, device, threshold=0.5):
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            probs   = torch.sigmoid(outputs.squeeze(1))

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    # --- threshold sweep ---
    print("Threshold sweep:")
    for t in [0.50, 0.30, 0.20, 0.10, 0.05]:
        predicted = (all_probs >= t).astype(float)
        cm        = confusion_matrix(all_labels, predicted)
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        f2          = fbeta_score(all_labels, predicted, beta=2)
        print(f"  t={t:.2f} | Sensitivity: {sensitivity:.1%} | "
              f"Specificity: {specificity:.1%} | "
              f"FN: {fn} | FP: {fp} | F2: {f2:.4f}")

    # --- full report at chosen threshold ---
    print(f"\nFull evaluation at t={threshold}:")
    predicted = (all_probs >= threshold).astype(float)
    cm        = confusion_matrix(all_labels, predicted)
    tn, fp, fn, tp = cm.ravel()

    auc         = roc_auc_score(all_labels, all_probs)
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f2          = fbeta_score(all_labels, predicted, beta=2)

    print(f"  AUC-ROC     : {auc:.4f}")
    print(f"  Accuracy    : {accuracy:.1%}")
    print(f"  Sensitivity : {sensitivity:.1%}")
    print(f"  Specificity : {specificity:.1%}")
    print(f"  F2 Score    : {f2:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives  (correctly said normal)  : {tn}")
    print(f"  False Positives (incorrectly flagged)     : {fp}")
    print(f"  False Negatives (missed abnormal)         : {fn} ← dangerous")
    print(f"  True Positives  (correctly caught)        : {tp}")
    print(f"\nClassification Report:")
    print(classification_report(all_labels, predicted,
                               target_names=["Normal", "Abnormal"]))

    return all_probs, all_labels

In [29]:
checkpoint = torch.load("checkpoints/best_model.pt")
model = ChestCNN().to(device) # just so we know what it is
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

cnn_probs, cnn_labels = evaluate_cnn(model, test_loader, device, threshold=0.5)


Threshold sweep:
  t=0.50 | Sensitivity: 67.5% | Specificity: 61.9% | FN: 158 | FP: 106 | F2: 0.6897
  t=0.30 | Sensitivity: 95.7% | Specificity: 13.7% | FN: 21 | FP: 240 | F2: 0.8777
  t=0.20 | Sensitivity: 99.8% | Specificity: 1.1% | FN: 1 | FP: 275 | F2: 0.8968
  t=0.10 | Sensitivity: 100.0% | Specificity: 0.0% | FN: 0 | FP: 278 | F2: 0.8973
  t=0.05 | Sensitivity: 100.0% | Specificity: 0.0% | FN: 0 | FP: 278 | F2: 0.8973

Full evaluation at t=0.5:
  AUC-ROC     : 0.7088
  Accuracy    : 65.4%
  Sensitivity : 67.5%
  Specificity : 61.9%
  F2 Score    : 0.6897

Confusion Matrix:
  True Negatives  (correctly said normal)  : 172
  False Positives (incorrectly flagged)     : 106
  False Negatives (missed abnormal)         : 158 ← dangerous
  True Positives  (correctly caught)        : 328

Classification Report:
              precision    recall  f1-score   support

      Normal       0.52      0.62      0.57       278
    Abnormal       0.76      0.67      0.71       486

    accuracy  

In [27]:
# run this cell to find your max stable batch size
import torch
import gc

def find_max_batch_size(model, device, image_size=224):
    test_sizes = [32, 64, 96, 128, 160, 192, 256, 320, 384, 512]
    
    last_working = 32
    for bs in test_sizes:
        try:
            torch.cuda.reset_peak_memory_stats()  # reset peak tracker
            torch.cuda.empty_cache()
            
            dummy        = torch.randn(bs, 1, image_size, image_size).to(device)
            dummy_labels = torch.zeros(bs).to(device)
            
            out  = model(dummy)
            loss = nn.BCEWithLogitsLoss()(out.squeeze(1), dummy_labels)
            loss.backward()
            model.zero_grad(set_to_none=True)
            
            # measure PEAK memory during forward+backward
            peak = torch.cuda.max_memory_allocated()/1e9
            curr = torch.cuda.memory_allocated()/1e9
            print(f"batch_size={bs:4d} ✓ | Peak VRAM: {peak:.2f} GB | "
                  f"Current: {curr:.2f} GB")
            last_working = bs
            
            del dummy, dummy_labels, out, loss
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"batch_size={bs:4d} ✗ | OOM")
                torch.cuda.empty_cache()
                break
            else:
                raise e
    
    # recommend leaving ~20% headroom for DataLoader buffers
    recommended = int(last_working * 0.8)
    print(f"\nMax working: {last_working} | Recommended (80%): {recommended}")
    return recommended

In [ ]:
raw_model = ChestCNN().to(device)
raw_model.apply(weights_init_cnn)
# this can take a second. don't do that. 
# best_bs = find_max_batch_size(raw_model, device)

batch_size=  32 ✓ | Peak VRAM: 1.17 GB | Current: 0.03 GB
batch_size=  64 ✓ | Peak VRAM: 2.23 GB | Current: 0.04 GB
batch_size=  96 ✓ | Peak VRAM: 3.33 GB | Current: 0.05 GB
batch_size= 128 ✓ | Peak VRAM: 4.42 GB | Current: 0.05 GB
batch_size= 160 ✓ | Peak VRAM: 5.52 GB | Current: 0.06 GB
batch_size= 192 ✓ | Peak VRAM: 6.62 GB | Current: 0.07 GB
batch_size= 256 ✓ | Peak VRAM: 8.82 GB | Current: 0.08 GB
batch_size= 320 ✓ | Peak VRAM: 11.01 GB | Current: 0.09 GB
batch_size= 384 ✓ | Peak VRAM: 13.21 GB | Current: 0.11 GB
batch_size= 512 ✓ | Peak VRAM: 17.60 GB | Current: 0.13 GB


KeyboardInterrupt: 